# Laboratorio 2

## Integrantes

- Sergio Orellana 221122
- Ricardo Chuy 221007
- Rodrigo Mansilla 22611

## Task 2

### 1. Para el MDP que diseñaron, ¿cuántas políticas deterministas posibles existen? Calculen el valor exacto y argumenten por qué la enumeración exhaustiva es inviable incluso para este problema relativamente pequeño.

_Respuesta:_

En mi diseño existen:

$$|\mathcal{S}|=1{,}000{,}000$$

estados y cuatro acciones formalmente disponibles en cada estado:

$$|\mathcal{A}|=4$$

Una política determinista estacionaria debe seleccionar exactamente una acción para cada estado. Por consiguiente, el número de políticas posibles es:

$$|\Pi_D|=|\mathcal{A}|^{|\mathcal{S}|}=4^{1{,}000{,}000}$$

El número resultante contiene **602,060 dígitos decimales**. Por lo tanto, no es viable almacenar, evaluar ni recorrer todas las políticas. Incluso si pudiera analizar mil millones de políticas por segundo, la enumeración seguiría requiriendo un tiempo incomparablemente mayor que cualquier escala operacional razonable.



In [1]:
# Esto solo sirve para verificación rápida de la cantidad de políticas deterministas posibles. 
import math

num_states = 5 * 2 * (5 ** 5) * (2 ** 5)
num_actions = 4
policy_digits = math.floor(num_states * math.log10(num_actions)) + 1

print("Número de estados:", num_states)
print("Número exacto de políticas: 4^1,000,000")
print("Cantidad de dígitos del número de políticas:", policy_digits)

Número de estados: 1000000
Número exacto de políticas: 4^1,000,000
Cantidad de dígitos del número de políticas: 602060


### 2. Dado que $\mathcal{T}^\pi$ es una contracción con factor $\gamma$, ¿cuántas iteraciones de Policy Evaluation necesitan teóricamente para reducir el error inicial a menos de $\theta=0.01$, asumiendo un error inicial de $\lVert V_0-V^\pi\rVert_\infty\leq \frac{R_{\max}}{1-\gamma}$? Desarrollen el cálculo usando la cota de convergencia geométrica y el valor de $\gamma$ que propusieron.

_Respuesta:_

Utilizo la cota de convergencia geométrica:

$$\lVert V_k-V^\pi\rVert_\infty
\leq
\gamma^k\lVert V_0-V^\pi\rVert_\infty$$

Como:

$$\lVert V_0-V^\pi\rVert_\infty
\leq
\frac{R_{\max}}{1-\gamma}$$

entonces:

$$\lVert V_k-V^\pi\rVert_\infty
\leq
\gamma^k\frac{R_{\max}}{1-\gamma}$$

Sustituyo $\gamma=0.95$, $R_{\max}=53$ y $\theta=0.01$:

$$0.95^k\frac{53}{1-0.95}<0.01$$

$$0.95^k\frac{53}{0.05}<0.01$$

$$0.95^k<\frac{0.01(0.05)}{53}$$

$$0.95^k<9.43396\times10^{-6}$$

Aplico logaritmos:

$$k>
\frac{\ln(9.43396\times10^{-6})}{\ln(0.95)}$$

$$k>225.59$$

Por consiguiente, necesito como mínimo:

$$k=226$$

iteraciones para garantizar teóricamente un error menor que $0.01$. Esta es una cota conservadora; por lo tanto, el algoritmo puede alcanzar el umbral antes en una ejecución concreta.


In [3]:
# Esta solo es una verificación rápida de la cantidad de iteraciones necesarias.
import math

gamma = 0.95
theta = 0.01
r_max = 53

k = math.ceil(
    math.log(theta * (1 - gamma) / r_max) / math.log(gamma)
)

error_k = (gamma ** k) * r_max / (1 - gamma)
error_previous = (gamma ** (k - 1)) * r_max / (1 - gamma)

print("Iteraciones mínimas:", k)
print("Cota del error en k:", error_k)
print("Cota del error en k - 1:", error_previous)


Iteraciones mínimas: 226
Cota del error en k: 0.009791306836437504
Cota del error en k - 1: 0.010306638775197373


### 3. Para su MDP específico, ¿anticipan que Policy Iteration o Value Iteration convergerá más rápido en términos de tiempo de cómputo total? Justifiquen su respuesta considerando el tamaño del espacio de estados, el número de acciones y el trade-off entre costo por iteración y número de iteraciones hasta convergencia.

_Respuesta:_

Para este MDP, anticipo que **Value Iteration será más conveniente en tiempo de cómputo total**. Mi espacio contiene $1{,}000{,}000$ estados y cuatro acciones por estado. Además, las transiciones son dispersas, ya que cada acción conduce solamente a un conjunto pequeño de estados sucesores posibles.

En cada barrido, Value Iteration aplica directamente el operador de optimalidad de Bellman:

$$V_{k+1}(s)=
\max_a
\sum_{s'}
p(s'\mid s,a)
\left[
r(s,a,s')+\gamma V_k(s')
\right]$$

Policy Iteration suele requerir menos iteraciones externas porque mejora una política completa en cada ciclo. Sin embargo, cada ciclo necesita evaluar la política actual mediante varios barridos de Bellman antes de ejecutar la mejora. Según la cota anterior, una evaluación exacta podría requerir hasta 226 barridos para alcanzar el umbral especificado.

Por consiguiente, considero que el menor costo de cada iteración de Value Iteration compensa su mayor número de iteraciones. No obstante, ninguno de los dos algoritmos tabulares sería directamente operativo con un millón de estados sin aprovechar representaciones dispersas, agregación de estados o aproximación de funciones. Si el modelo se redujera considerablemente, Policy Iteration podría superar a Value Iteration debido a su menor número de mejoras externas.
